# Smart WasteVision — Waste Image Classification (TrashNet)

This notebook trains and compares CNN models that classify waste images into six
categories — **cardboard, glass, metal, paper, plastic, trash** — using the
[TrashNet](https://github.com/garythung/trashnet) dataset.

**Pipeline:** dataset loading & cleaning → class distribution → stratified
train/val/test split → transforms → `Dataset`/`DataLoader` → visualization →
CNN model → training with best-checkpoint saving → validation → classification
report & confusion matrix → class-weighted loss experiment → final test
evaluation → model saving → single-image prediction.

> Run every cell top to bottom in Google Colab. No manual edits are required.

## 1. Imports & Configuration

In [ ]:
# Core
import os
import random
import shutil
import hashlib
import zipfile
import json as _json
from pathlib import Path
from collections import Counter

# Data
import numpy as np
import pandas as pd

# Viz
import matplotlib.pyplot as plt
from PIL import Image

# ML / DL
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------------------------
# Fixed class mapping (BUG FIX #6 — never re-derive this from sorted folder
# names inside a function; every part of the notebook imports it from here).
# ---------------------------------------------------------------------------
CLASS_TO_IDX = {
    "cardboard": 0,
    "glass": 1,
    "metal": 2,
    "paper": 3,
    "plastic": 4,
    "trash": 5,
}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}
CLASSES = list(CLASS_TO_IDX.keys())
NUM_CLASSES = len(CLASSES)

# ---------------------------------------------------------------------------
# General config
# ---------------------------------------------------------------------------
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Classes:", CLASS_TO_IDX)


## 2. Dataset Loading

In [ ]:
# Clone TrashNet (skip if already cloned — safe to re-run this cell).
if not Path("trashnet").exists():
    !git clone https://github.com/garythung/trashnet.git

zip_path = Path("trashnet/data/dataset-resized.zip")
raw_dataset_path = Path("trashnet/data/dataset-resized")

print("ZIP exists:", zip_path.exists())
print("Already extracted:", raw_dataset_path.exists())


In [ ]:
# Extract only if not already extracted.
if not raw_dataset_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(zip_path.parent)
    print("Dataset extracted successfully!")
else:
    print("Dataset already extracted — skipping.")

print("Dataset exists:", raw_dataset_path.exists())


In [ ]:
raw_class_counts = {}

for folder in sorted(raw_dataset_path.iterdir()):
    if folder.is_dir():
        raw_class_counts[folder.name] = len(list(folder.glob("*")))

print("Raw (uncleaned) class counts:")
for class_name, count in raw_class_counts.items():
    print(f"{class_name:10} -> {count} images")
print("Total:", sum(raw_class_counts.values()))


### 2.1 Remove duplicate images

TrashNet contains a handful of exact duplicate files (same image saved twice,
sometimes in different classes). We detect duplicates by file hash and build a
**clean** copy of the dataset that keeps only the first occurrence of each
image (**BUG FIX** — the original notebook hardcoded three duplicate file
paths by hand; here duplicates are detected and removed programmatically, so
the step is reproducible even if the raw archive changes).

In [ ]:
image_hashes = {}     # hash -> first (kept) path
duplicate_paths = []  # paths to be dropped

for class_name in sorted(raw_class_counts.keys()):
    for image_path in sorted((raw_dataset_path / class_name).glob("*")):
        with open(image_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        if file_hash in image_hashes:
            duplicate_paths.append(image_path)
        else:
            image_hashes[file_hash] = image_path

print("Total raw images   :", sum(raw_class_counts.values()))
print("Duplicate images   :", len(duplicate_paths))
print("Unique images kept :", len(image_hashes))


In [ ]:
clean_path = Path("trashnet/clean_dataset")

if clean_path.exists():
    shutil.rmtree(clean_path)

for class_name in raw_class_counts:
    (clean_path / class_name).mkdir(parents=True, exist_ok=True)

duplicate_set = set(str(p) for p in duplicate_paths)

for class_name in sorted(raw_class_counts.keys()):
    for image_path in sorted((raw_dataset_path / class_name).glob("*")):
        if str(image_path) in duplicate_set:
            continue
        shutil.copy(image_path, clean_path / class_name / image_path.name)

clean_class_counts = {
    folder.name: len(list(folder.glob("*")))
    for folder in sorted(clean_path.iterdir()) if folder.is_dir()
}

print("Clean class counts:")
for class_name, count in clean_class_counts.items():
    print(f"{class_name:10} -> {count} images")
print("Total:", sum(clean_class_counts.values()))


## 3. Class Distribution

In [ ]:
data = []
for class_name in CLASSES:
    class_folder = clean_path / class_name
    for image_path in sorted(class_folder.glob("*")):
        data.append({"image_path": str(image_path), "label": class_name})

df = pd.DataFrame(data)

print("Total images:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts().reindex(CLASSES))
df.head()


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(df["label"].value_counts().reindex(CLASSES).index,
        df["label"].value_counts().reindex(CLASSES).values)
plt.xlabel("Waste Class")
plt.ylabel("Number of Images")
plt.title("Class Distribution (Cleaned Dataset)")
plt.show()


## 4. Stratified Train / Validation / Test Split

In [ ]:
# 70% train, 15% validation, 15% test — stratified on the label so every
# split keeps the original class proportions.
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED,
)

# BUG FIX #1 — reset every split's index after train_test_split so later
# `.loc[index, ...]` lookups inside the Dataset are never off by the
# original DataFrame's row numbers.
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


In [ ]:
print("Train distribution:")
print(train_df["label"].value_counts().reindex(CLASSES))

print("\nValidation distribution:")
print(val_df["label"].value_counts().reindex(CLASSES))

print("\nTest distribution:")
print(test_df["label"].value_counts().reindex(CLASSES))


## 5. Transforms

In [ ]:
# Training transform: light augmentation + ImageNet normalization.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Validation / test transform: deterministic resize + the SAME normalization
# used for training (BUG FIX — the original notebook redefined this
# transform twice, once without normalization, causing train/eval mismatch).
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


## 6. Dataset & DataLoader

In [ ]:
class WasteDataset(Dataset):
    """Loads waste images from a DataFrame with `image_path` / `label` columns."""

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_path = self.dataframe.loc[index, "image_path"]
        label_name = self.dataframe.loc[index, "label"]

        image = Image.open(image_path).convert("RGB")
        label = CLASS_TO_IDX[label_name]

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
train_dataset = WasteDataset(train_df, transform=train_transform)
val_dataset = WasteDataset(val_df, transform=val_test_transform)
test_dataset = WasteDataset(test_df, transform=val_test_transform)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

image, label = train_dataset[0]
print("\nSample image shape:", image.shape)
print("Sample label:", label, "->", IDX_TO_CLASS[label])


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

images, labels = next(iter(train_loader))
print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)


## 7. Visualization

In [ ]:
def denormalize(tensor_image):
    """Undo ImageNet normalization for display purposes."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    image = tensor_image.cpu() * std + mean
    return image.clamp(0, 1).permute(1, 2, 0).numpy()


# One random raw (un-augmented) example per class.
plt.figure(figsize=(15, 8))
for i, class_name in enumerate(CLASSES):
    image_path = random.choice(list((clean_path / class_name).glob("*")))
    image = Image.open(image_path)

    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.title(class_name)
    plt.axis("off")
plt.suptitle("One sample per class")
plt.tight_layout()
plt.show()


In [ ]:
# A batch straight out of the training loader (shows augmentation in action).
images, labels = next(iter(train_loader))

plt.figure(figsize=(15, 8))
for i in range(min(8, len(images))):
    plt.subplot(2, 4, i + 1)
    plt.imshow(denormalize(images[i]))
    plt.title(IDX_TO_CLASS[labels[i].item()])
    plt.axis("off")
plt.suptitle("Augmented training batch")
plt.tight_layout()
plt.show()


## 8. CNN Model

In [ ]:
class WasteCNN(nn.Module):
    """Compact CNN with BatchNorm + Dropout.

    Uses `AdaptiveAvgPool2d` before the classifier head so the flattened
    feature size never depends on the exact input resolution
    (BUG FIX — the original model hardcoded `128 * 28 * 28`, which silently
    breaks if `IMG_SIZE` ever changes).
    """

    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Quick shape sanity check.
_sample_model = WasteCNN().to(DEVICE)
images, labels = next(iter(train_loader))
outputs = _sample_model(images.to(DEVICE))
print("Input shape :", images.shape)
print("Output shape:", outputs.shape)
del _sample_model


## 9. Training Utilities (train / validate / checkpoint)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


def train_model(model, criterion, optimizer, train_loader, val_loader,
                 device, num_epochs, checkpoint_path):
    """Trains `model` and saves the best validation checkpoint.

    BUG FIX #2/#3/#4 — the SAME `model`, `criterion`, and `optimizer` are
    threaded through every call in this function (no other model is ever
    trained or evaluated here), and the best validation checkpoint is saved
    to disk and reloaded before returning.
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            torch.save(model.state_dict(), checkpoint_path)

        flag = " (best so far -> checkpoint saved)" if improved else ""
        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"| Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}{flag}"
        )

    # Reload the best checkpoint before handing the model back.
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"\nBest validation accuracy: {best_val_acc:.4f} (loaded into model)")

    return model, history


@torch.no_grad()
def evaluate_on_test(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        predictions = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(predictions)
        all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds)


def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm)
    plt.xticks(range(NUM_CLASSES), CLASSES, rotation=45)
    plt.yticks(range(NUM_CLASSES), CLASSES)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.colorbar()

    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.tight_layout()
    plt.show()
    return cm


## 10. Experiment A — Standard `CrossEntropyLoss`

In [ ]:
model_standard = WasteCNN(num_classes=NUM_CLASSES).to(DEVICE)
criterion_standard = nn.CrossEntropyLoss()
optimizer_standard = torch.optim.Adam(model_standard.parameters(), lr=LEARNING_RATE)

model_standard, history_standard = train_model(
    model=model_standard,
    criterion=criterion_standard,
    optimizer=optimizer_standard,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    checkpoint_path="best_model_standard.pth",
)


### Evaluate Experiment A on the test set

In [ ]:
y_true_standard, y_pred_standard = evaluate_on_test(model_standard, test_loader, DEVICE)

acc_standard = accuracy_score(y_true_standard, y_pred_standard)
f1_macro_standard = f1_score(y_true_standard, y_pred_standard, average="macro")

print(f"Test Accuracy : {acc_standard:.4f} ({acc_standard * 100:.2f}%)")
print(f"Macro F1      : {f1_macro_standard:.4f}")
print()
print(classification_report(y_true_standard, y_pred_standard, target_names=CLASSES))

cm_standard = plot_confusion_matrix(
    y_true_standard, y_pred_standard, "Confusion Matrix - Standard CrossEntropyLoss"
)


## 11. Class Weights (computed directly from the training DataFrame)

In [ ]:
# BUG FIX #5 — class weights come straight from `train_df["label"]` value
# counts (no iteration over the Dataset/DataLoader is needed).
train_label_counts = train_df["label"].value_counts().reindex(CLASSES)
total_train = len(train_df)

class_weights = torch.tensor(
    [total_train / (NUM_CLASSES * train_label_counts[c]) for c in CLASSES],
    dtype=torch.float32,
).to(DEVICE)

print("Training class counts:")
print(train_label_counts)
print("\nClass weights:")
for class_name, weight in zip(CLASSES, class_weights):
    print(f"{class_name:10} -> {weight.item():.4f}")


## 12. Experiment B — Class-Weighted `CrossEntropyLoss`

In [ ]:
model_weighted = WasteCNN(num_classes=NUM_CLASSES).to(DEVICE)
criterion_weighted = nn.CrossEntropyLoss(weight=class_weights)
optimizer_weighted = torch.optim.Adam(model_weighted.parameters(), lr=LEARNING_RATE)

model_weighted, history_weighted = train_model(
    model=model_weighted,
    criterion=criterion_weighted,
    optimizer=optimizer_weighted,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    checkpoint_path="best_model_weighted.pth",
)


### Evaluate Experiment B on the test set

In [ ]:
y_true_weighted, y_pred_weighted = evaluate_on_test(model_weighted, test_loader, DEVICE)

acc_weighted = accuracy_score(y_true_weighted, y_pred_weighted)
f1_macro_weighted = f1_score(y_true_weighted, y_pred_weighted, average="macro")

print(f"Test Accuracy : {acc_weighted:.4f} ({acc_weighted * 100:.2f}%)")
print(f"Macro F1      : {f1_macro_weighted:.4f}")
print()
print(classification_report(y_true_weighted, y_pred_weighted, target_names=CLASSES))

cm_weighted = plot_confusion_matrix(
    y_true_weighted, y_pred_weighted, "Confusion Matrix - Class-Weighted CrossEntropyLoss"
)


## 13. Final Comparison

In [ ]:
comparison_df = pd.DataFrame({
    "Experiment": ["Standard CrossEntropyLoss", "Class-Weighted CrossEntropyLoss"],
    "Test Accuracy": [acc_standard, acc_weighted],
    "Macro F1": [f1_macro_standard, f1_macro_weighted],
})

print(comparison_df.to_string(index=False))

per_class_rows = []
for class_name in CLASSES:
    idx = CLASS_TO_IDX[class_name]
    report_standard = classification_report(
        y_true_standard, y_pred_standard, target_names=CLASSES, output_dict=True
    )[class_name]
    report_weighted = classification_report(
        y_true_weighted, y_pred_weighted, target_names=CLASSES, output_dict=True
    )[class_name]

    per_class_rows.append({
        "class": class_name,
        "precision_standard": report_standard["precision"],
        "recall_standard": report_standard["recall"],
        "f1_standard": report_standard["f1-score"],
        "precision_weighted": report_weighted["precision"],
        "recall_weighted": report_weighted["recall"],
        "f1_weighted": report_weighted["f1-score"],
    })

per_class_df = pd.DataFrame(per_class_rows).round(4)
per_class_df


## 14. Save Final Model

In [ ]:
# Pick whichever experiment has the higher macro F1 as the deployed model.
if f1_macro_weighted >= f1_macro_standard:
    best_experiment_name = "Class-Weighted CrossEntropyLoss"
    best_checkpoint_path = "best_model_weighted.pth"
    best_model = model_weighted
else:
    best_experiment_name = "Standard CrossEntropyLoss"
    best_checkpoint_path = "best_model_standard.pth"
    best_model = model_standard

FINAL_MODEL_PATH = "waste_classifier_final.pth"
shutil.copy(best_checkpoint_path, FINAL_MODEL_PATH)

with open("class_mapping.json", "w") as f:
    _json.dump({"class_to_idx": CLASS_TO_IDX, "idx_to_class": IDX_TO_CLASS}, f, indent=2)

print(f"Best experiment : {best_experiment_name}")
print(f"Saved model     : {FINAL_MODEL_PATH}")
print("Saved mapping   : class_mapping.json")


## 15. Single-Image Prediction

In [ ]:
def predict_image(model, image_path, transform, device, idx_to_class=IDX_TO_CLASS):
    """Runs the trained model on a single image and returns (class_name, confidence)."""
    model.eval()

    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        predicted_idx = int(torch.argmax(probabilities).item())
        confidence = float(probabilities[predicted_idx].item())

    predicted_class = idx_to_class[predicted_idx]

    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(f"Predicted: {predicted_class} ({confidence * 100:.1f}%)")
    plt.axis("off")
    plt.show()

    print("Class probabilities:")
    for i, class_name in enumerate(CLASSES):
        print(f"{class_name:10} -> {probabilities[i].item():.4f}")

    return predicted_class, confidence


# Demo: run the saved best model on a random test-set image.
best_model.load_state_dict(torch.load(FINAL_MODEL_PATH, map_location=DEVICE))
demo_image_path = test_df.sample(1, random_state=SEED)["image_path"].iloc[0]

predicted_class, confidence = predict_image(
    best_model, demo_image_path, val_test_transform, DEVICE
)
print(f"\nPredicted class: {predicted_class} | Confidence: {confidence:.4f}")
